# Photothermal Parameter Estimation: PINN vs Classical Methods


In [ ]:
import os
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')  # headless mode for scripts

import functools
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import seaborn as sns
import pandas as pd
from scipy.optimize import curve_fit

from network import Net, NetDiscovery, np_to_th, DEVICE
from diff_equations import grad, cooling_law

sns.set_theme()
torch.manual_seed(42)
torch.set_num_threads(4)

In [ ]:
# Disk cache for the PINN training runs: a run with the same data and settings is loaded from cache/ instead of trained again.
# USE_CACHE = False trains everything again; raise CACHE_VERSION (or delete cache/) after changing a training function.
import hashlib

USE_CACHE = True
CACHE_VERSION = 1
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)


def cache_file(name, *parts):
    h = hashlib.md5(str(CACHE_VERSION).encode())
    for p in parts:
        h.update(np.asarray(p, dtype=float).tobytes() if isinstance(p, np.ndarray) else repr(p).encode())
    return os.path.join(CACHE_DIR, f"{name}_{h.hexdigest()}.pt")

## 2 · Physical Model
### Governing ODE
### Analytic solution
### Roper bias


## 3 · Analytic Solution and Synthetic Data


In [ ]:
def heating_cooling(t, a, tau, t_off, Tenv=25.0):
    """Exact analytic solution of the photothermal ODE."""
    theta_off = a * tau * (1 - np.exp(-t_off / tau))
    return Tenv + np.where(
        t < t_off,
        a * tau * (1 - np.exp(-t / tau)),
        theta_off * np.exp(-(t - t_off) / tau),
    )

In [ ]:
# True parameter values (to be recovered by each method)
np.random.seed(1)
Tenv_true  = 25.0    # ambient temperature [°C]
a_true     = 0.05    # heating rate [°C/s]
tau_true   = 300.0   # thermal time constant [s]
t_off_true = 1 * tau_true # laser switch-off time [s]  (short experiment: t_off/τ = 1)

times = np.arange(0, 3 * tau_true, 5)
eq    = functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=t_off_true, Tenv=Tenv_true)
temps = eq(times)

# Training data: 10 s sampling interval, 0.1 °C Gaussian noise
t = np.arange(0, 3 * tau_true + 5, 10)
T = eq(t) + 0.1 * np.random.randn(len(t))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(times / 60, temps, lw=2, label='Exact solution')
ax.plot(t / 60, T, 'o', ms=3, alpha=0.6, label='Simulated data (0.1 °C noise)')
ax.axvline(t_off_true / 60, color='gray', ls='--', lw=0.8, label='laser off')
ax.set_xlabel('Time (min)'); ax.set_ylabel('Temperature (°C)')
ax.set_title(f'Synthetic experiment — $t_{{\\mathrm{{off}}}}/\\tau = {t_off_true/tau_true:.0f}$')
ax.legend()
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig0_synthetic_data.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 4 · Classical Baselines
### 4.1 Roper method
### 4.2 Whole-curve least-squares


In [ ]:
def roper(t, T, t_off, Tenv_=Tenv_true, min_frac=0.05):
    """Standard Roper et al. 2007 method."""
    theta  = T - Tenv_
    dT_max = theta[t <= t_off].max()
    cool   = (t > t_off) & (theta > min_frac * dT_max)
    slope, _ = np.polyfit(t[cool] - t_off, np.log(theta[cool]), 1)
    tau_est  = -1.0 / slope
    return dT_max / tau_est, tau_est

In [ ]:
def whole_curve_fit(t, T, t_off, Tenv_=Tenv_true):
    """Least-squares fit to the full analytic ODE solution."""
    f = lambda t_, a_, tau_: heating_cooling(t_, a_, tau_, t_off, Tenv_)
    (a_est, tau_est), _ = curve_fit(f, t, T, p0=(0.01, 100.0),
                                    bounds=([1e-6, 1.0], [10.0, 1e5]))
    return a_est, tau_est

In [ ]:
a_r,  tau_r  = roper(t, T, t_off_true)
a_wc, tau_wc = whole_curve_fit(t, T, t_off_true)

print(f"True values :  a = {a_true:.4f} °C/s    τ = {tau_true:.0f} s")
print(f"Roper       :  a = {a_r:.4f} °C/s  ({100*(a_r-a_true)/a_true:+.1f} %)    τ = {tau_r:.0f} s")
print(f"Whole-curve :  a = {a_wc:.4f} °C/s  ({100*(a_wc-a_true)/a_true:+.1f} %)    τ = {tau_wc:.0f} s")

## 5 · Physics-Informed Neural Network (PINN)


In [ ]:
class PhotothermalPINN(NetDiscovery):
    """PINN for discovering (a, τ) from photothermal heating–cooling data."""

    def __init__(self, t_off_hat, switch_input=True,
                 n_units=100, epochs=3000, data_loss=nn.MSELoss(),
                 lr=3e-3, physics_loss=None, data_weight=1.0, physics_weight=1.0):
        n_in = 2 if switch_input else 1
        super().__init__(input_dim=n_in, output_dim=1,
                         n_units=n_units, epochs=epochs,
                         data_loss=data_loss, lr=lr,
                         physics_loss=physics_loss,
                         data_weight=data_weight, physics_weight=physics_weight)

        # Replace ReLU activations with tanh for smooth ODE gradients
        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.ReLU):
                self.layers[i] = nn.Tanh()

        self.t_off_hat    = t_off_hat
        self.switch_input = switch_input
        self.a            = nn.Parameter(data=torch.tensor([0.]))

    def forward(self, t):
        if self.switch_input:
            t = torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1)
        return self.out(self.layers(t))

In [ ]:
def train_photothermal(t_, T_, t_off_, Tenv_=Tenv_true,
                       switch_input=True, data_weight=1.0, physics_weight=1.0,
                       adam_epochs=3000, lbfgs_iters=300, seed=0, return_fit=False, log=None):
    """Train PhotothermalPINN and return (a_est, tau_est), plus the network's fitted T if return_fit."""
    torch.manual_seed(seed)

    # Normalise inputs to [0, 1]
    t_scale = float(t_.max())
    T_scale = float(max((T_ - Tenv_).max(), 1e-3))
    t_hat, theta_hat = t_ / t_scale, (T_ - Tenv_) / T_scale
    t_off_hat    = t_off_ / t_scale

    def physics_loss(model):
        t_col = torch.linspace(0, 1, 800).view(-1, 1).requires_grad_(True).to(DEVICE)
        laser = (t_col < t_off_hat).float()
        temps = model(t_col)
        dT    = grad(temps, t_col)[0]
        pde   = model.a * laser - model.kappa * temps - dT
        ic    = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return torch.mean(pde ** 2) + ic

    model = PhotothermalPINN(t_off_hat, switch_input=switch_input,
                              physics_loss=physics_loss,
                              data_weight=data_weight, physics_weight=physics_weight,
                              epochs=adam_epochs, lr=3e-3).to(DEVICE)   # same device as the data
    Xt_ = np_to_th(t_hat.reshape(-1, 1))
    yt_ = np_to_th(theta_hat.reshape(-1, 1))
    cache_path = cache_file("photothermal", t_, T_, t_off_, Tenv_, switch_input, data_weight, physics_weight, adam_epochs, lbfgs_iters, seed)
    if USE_CACHE and log is None and os.path.exists(cache_path):
        model.load_state_dict(torch.load(cache_path, map_location=DEVICE))      # trained before: no training
    else:
        if log is not None:
            log.start(model, T_scale / t_scale)
        model.fit(t_hat.reshape(-1, 1), theta_hat.reshape(-1, 1),
                  on_step=None if log is None else lambda ep, d, p: log.record("Adam", d, p))
        if log is not None:
            log.end_adam()

        # L-BFGS fine-tuning
        opt = torch.optim.LBFGS(model.parameters(), lr=1.0,
                                  max_iter=lbfgs_iters, line_search_fn='strong_wolfe')
        def _closure():
            opt.zero_grad()
            data_term = model.data_loss(yt_, model(Xt_))
            physics_term = physics_loss(model)
            l = data_weight * data_term + physics_weight * physics_term
            l.backward()
            if log is not None:
                log.record("L-BFGS", data_term, physics_term)
            return l
        opt.step(_closure)
        if log is not None:
            log.finish(opt)
        if USE_CACHE and log is None:
            torch.save(model.state_dict(), cache_path)

    a_est, tau_est = model.a.item() * T_scale / t_scale, t_scale / model.kappa.item()
    if return_fit:   # the network's own output at the data times, in °C
        fit = model(Xt_).detach().cpu().numpy().ravel() * T_scale + Tenv_
        return a_est, tau_est, fit
    return a_est, tau_est

## 6 · Experiment 1 — Full vs Short Heating Duration

In [ ]:
def make_experiment(t_off_e, t_end, dt=10, noise=0.1, seed=0):
    rng = np.random.default_rng(seed)
    t_e = np.arange(0, t_end + dt / 2, dt)
    T_e = functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=t_off_e, Tenv=Tenv_true)(t_e)
    return t_e, T_e + rng.normal(0, noise, len(t_e))

EXPERIMENTS = {
    "full  (heat 5τ, cool 5τ)":  dict(t_off_e=5 * tau_true, t_end=10 * tau_true),
    "short (heat 1τ, cool 2τ)": dict(t_off_e=1 * tau_true, t_end= 3 * tau_true),
}

In [ ]:
METHODS = {
    "Roper":           roper,
    "whole-curve fit": whole_curve_fit,
    "PINN (no switch)": lambda t_, T_, to: train_photothermal(t_, T_, to, switch_input=False),
    "PINN + switch":    lambda t_, T_, to: train_photothermal(t_, T_, to, switch_input=True),
}

rows, fits = [], {}
for exp_name, e in EXPERIMENTS.items():
    te, Te = make_experiment(**e, seed=1)
    for m_name, fn in METHODS.items():
        a_, tau_ = fn(te, Te, e['t_off_e'])
        fits[(exp_name, m_name)] = (a_, tau_)
        rows.append(dict(experiment=exp_name, method=m_name,
                         a_err=round(100 * (a_ - a_true) / a_true, 1),
                         tau_err=round(100 * (tau_ - tau_true) / tau_true, 1)))

df1 = pd.DataFrame(rows)
print("=== Error in a [%] ===")
print(df1.pivot(index='method', columns='experiment', values='a_err').to_string())

### Figure 1 — Fitted curves on the short experiment


In [ ]:
e_sh = EXPERIMENTS["short (heat 1τ, cool 2τ)"]
t_sh, T_sh = make_experiment(**e_sh, seed=1)
tt = np.linspace(0, t_sh.max(), 1000)

styles = {
    "Roper":           dict(color='tab:orange', ls='--', lw=1.8),
    "whole-curve fit": dict(color='tab:green',  ls='-.', lw=1.8),
    "PINN (no switch)": dict(color='tab:red',   ls=':',  lw=1.8),
    "PINN + switch":   dict(color='tab:blue',   ls='-',  lw=2.0),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_sh / 60, T_sh, '.', ms=3, color='lightgray', zorder=1, label='data')
ax.plot(tt / 60, eq(tt), 'k-', lw=1.2, alpha=0.5, label='true')

for m_name, sty in styles.items():
    a_e, tau_e = fits[("short (heat 1τ, cool 2τ)", m_name)]
    err = 100 * (a_e - a_true) / a_true
    ax.plot(tt / 60,
            heating_cooling(tt, a_e, tau_e, e_sh['t_off_e'], Tenv_true),
            label=f'{m_name}  (Δa={err:+.1f}%)', **sty)

ax.axvline(e_sh['t_off_e'] / 60, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Temperature (°C)')
ax.set_title(r'Short experiment ($t_{\mathrm{off}}/\tau = 1$) — method comparison')
ax.legend(fontsize=8)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig1_short_experiment.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 7 · Why the Switch Input Matters

In [ ]:
zoom_lo = 0.7 * e_sh['t_off_e']
zoom_hi = 1.5 * e_sh['t_off_e']
tt_w    = np.linspace(zoom_lo, zoom_hi, 600)
mask    = (t_sh > zoom_lo) & (t_sh < zoom_hi)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(t_sh[mask] / 60, T_sh[mask], '.', ms=4, color='lightgray', label='data')
ax.plot(tt_w / 60, eq(tt_w), 'k-', lw=1.2, alpha=0.5, label='true')

for key in ['PINN (no switch)', 'PINN + switch']:
    a_e, tau_e = fits[("short (heat 1τ, cool 2τ)", key)]
    ax.plot(tt_w / 60,
            heating_cooling(tt_w, a_e, tau_e, e_sh['t_off_e'], Tenv_true),
            label=key, **styles[key])

ax.axvline(e_sh['t_off_e'] / 60, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Zoom around laser switch-off')
ax.legend(fontsize=9)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig2_switch_zoom.jpg', dpi=150, bbox_inches='tight')
plt.show()


### 7.1 Is the switch effect reproducible?

In [ ]:
# Section 7.1: same data as Section 6, five initialisation seeds, with and without the switch input
from contextlib import redirect_stdout
import io

abl_rows = []
for exp_name, e in EXPERIMENTS.items():
    te, Te = make_experiment(**e, seed=1)
    for switch in (False, True):
        for seed_ in range(5):
            with redirect_stdout(io.StringIO()):          # hide the per-epoch log
                a_, tau_ = train_photothermal(te, Te, e['t_off_e'], switch_input=switch, seed=seed_)
            abl_rows.append(dict(experiment=exp_name, switch=switch, seed=seed_,
                                 a_err_pct=100 * (a_ - a_true) / a_true,
                                 tau_err_pct=100 * (tau_ - tau_true) / tau_true))

df_abl = pd.DataFrame(abl_rows)
os.makedirs("results", exist_ok=True)
df_abl.to_csv("results/switch_ablation.csv", index=False)
print("=== Error in a [%] over five initialisation seeds (same data) ===")
print(df_abl.groupby(['experiment', 'switch'])['a_err_pct'].agg(['mean', 'std', 'min', 'max']).round(2).to_string())

## 8 · Validation on Real Experimental Data

In [ ]:
import os

REAL_DATA_DIR = "real_data"

# t_off and Tenv come from the digitised curves (time origin at laser switch-on);
# ref_a and ref_tau are the whole-curve least-squares fit, used as the reference on real data.
REAL_DATASETS = {
    "Vikas 2023\nGNR 10×38 nm, 808 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "gnr_10x38_20ugml_NIR_laser.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=900.0, Tenv=25.0, ref_a=0.04561, ref_tau=381.0,
    ),
    "Sánchez 2014\nB-GNR 10×41 nm, 808 nm": dict(
        file        = os.path.join(REAL_DATA_DIR, "gnr_10x41_peg_bare_808nm_2W.csv"),
        nanoparticle= "B-GNR_10x41nm_808nm_2W",
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1799.2, Tenv=25.47, ref_a=0.08740, ref_tau=228.4,
    ),
    "Sánchez 2014\nPEG-GNR 10×41 nm, 808 nm": dict(
        file        = os.path.join(REAL_DATA_DIR, "gnr_10x41_peg_bare_808nm_2W.csv"),
        nanoparticle= "PEG-GNR_10x41nm_808nm_2W",
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1799.2, Tenv=25.34, ref_a=0.11486, ref_tau=220.2,
    ),
    "Armenta-Gamez 2026\nAuNR, 785 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "aunr_785nm_1p2Wcm2_pmc13028931.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1347.4, Tenv=23.85, ref_a=0.10533, ref_tau=377.0,
    ),
    "Armenta-Gamez 2026\nAuNR@Si, 785 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "aunrsi_785nm_1p2Wcm2_pmc13028931.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1255.1, Tenv=24.15, ref_a=0.09278, ref_tau=220.9,
    ),
}

def load_real(ds):
    df = pd.read_csv(ds["file"])
    if "nanoparticle" in ds:
        df = df[df["nanoparticle"] == ds["nanoparticle"]]
    return df[ds["t_col"]].values.astype(float), df[ds["T_col"]].values.astype(float)

In [ ]:
real_rows, real_fits = [], {}

METHOD_FNS = [
    ("Roper",           lambda t_, T_, to, Tv: roper(t_, T_, to, Tenv_=Tv)),
    ("whole-curve fit", lambda t_, T_, to, Tv: whole_curve_fit(t_, T_, to, Tenv_=Tv)),
    ("PINN + switch",   lambda t_, T_, to, Tv: train_photothermal(t_, T_, to, Tenv_=Tv)),
]

for ds_name, ds in REAL_DATASETS.items():
    t_r, T_r   = load_real(ds)
    t_off_r    = ds["t_off"]
    Tenv_r     = ds["Tenv"]
    ref_a, ref_tau = ds["ref_a"], ds["ref_tau"]

    label = ds_name.replace('\n', ' | ')
    print(f"\n{'─'*60}\n  {label}")
    print(f"  ref: a={ref_a:.4f} °C/s   τ={ref_tau:.0f} s   t_off/τ={t_off_r/ref_tau:.1f}")

    for m_name, fn in METHOD_FNS:
        a_e, tau_e = fn(t_r, T_r, t_off_r, Tenv_r)
        da = 100 * (a_e - ref_a) / ref_a
        dt = 100 * (tau_e - ref_tau) / ref_tau
        rms = np.sqrt(np.mean((T_r - heating_cooling(t_r, a_e, tau_e, t_off_r, Tenv_r)) ** 2))
        print(f"  {m_name:22s}  a={a_e:.4f} ({da:+.1f}%)   τ={tau_e:.0f} ({dt:+.1f}%)   RMS={rms:.2f} °C")
        real_fits[(ds_name, m_name)] = (a_e, tau_e)
        real_rows.append(dict(
            dataset=ds_name.replace('\n', ' '), method=m_name,
            a_est=round(a_e, 5), tau_est=round(tau_e, 1),
            a_err_pct=round(da, 1), tau_err_pct=round(dt, 1), rms_C=round(rms, 2),
            t_off_over_tau=round(t_off_r / ref_tau, 2),
        ))

df_real = pd.DataFrame(real_rows)
os.makedirs("results", exist_ok=True)
df_real.to_csv("results/real_data_results.csv", index=False)

print("\n=== Summary: error in a [%] ===")
print(df_real.pivot(index='method', columns='dataset', values='a_err_pct').to_string())
print("\n=== Residual RMS of the fitted ODE curve [°C] ===")
print(df_real.pivot(index='method', columns='dataset', values='rms_C').to_string())

### Figure 2 — Real data: method comparison


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()
axes_flat[-1].set_visible(False)

method_styles = {
    "Roper":           dict(color='tab:orange', ls='--', lw=1.8),
    "whole-curve fit": dict(color='tab:green',  ls='-.', lw=1.8),
    "PINN + switch":   dict(color='tab:blue',   ls=':',  lw=2.2),
}

for ax, (ds_name, ds) in zip(axes_flat, REAL_DATASETS.items()):
    t_r, T_r    = load_real(ds)
    t_off_r     = ds["t_off"]
    Tenv_r      = ds["Tenv"]
    ref_a, ref_tau = ds["ref_a"], ds["ref_tau"]
    tt = np.linspace(0, t_r.max(), 1000)

    ax.plot(t_r / 60, T_r, color='lightgray', lw=1.0, zorder=1, label='data')
    ax.plot(tt / 60, heating_cooling(tt, ref_a, ref_tau, t_off_r, Tenv_r),
            'k--', lw=1.2, alpha=0.6,
            label=f'ref  a={ref_a:.4f}, τ={ref_tau:.0f} s', zorder=2)

    for m_name, sty in method_styles.items():
        a_e, tau_e = real_fits[(ds_name, m_name)]
        da = 100 * (a_e - ref_a) / ref_a
        ax.plot(tt / 60, heating_cooling(tt, a_e, tau_e, t_off_r, Tenv_r),
                label=f'{m_name}  ({da:+.1f}%)', zorder=3, **sty)

    ax.axvline(t_off_r / 60, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('T (°C)')
    ax.set_title(ds_name.replace('\n', '\n'), fontsize=9)
    ax.legend(fontsize=6.5)

plt.suptitle('Real data validation — error in $a$ relative to the whole-curve reference', fontsize=11)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig4_real_data.jpg', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Nonlinear Heat Loss: Learning the Loss Law

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares


def simulate_loss(t, a_, f, t_off_):
    # numerical solution of dθ/dt = a L(t) - f(θ), θ(0) = 0
    rhs = lambda tt, y: [a_ * (tt < t_off_) - f(max(y[0], 0.0))]
    return solve_ivp(rhs, (0, t.max()), [0.0], t_eval=t, max_step=5.0, rtol=1e-8, atol=1e-10).y[0]

In [ ]:
def single_tau_fit(t, th, t_off_):
    res = least_squares(lambda p: simulate_loss(t, p[0], lambda x: p[1] * x, t_off_) - th, [0.05, 3e-3],
                        bounds=([1e-5, 1e-6], [5, 1]), diff_step=1e-3, x_scale="jac")
    return res.x, np.sqrt(np.mean(res.fun ** 2))

In [ ]:
def power_law_fit(t, th, t_off_):
    # f(θ) = k_ref·θref·(θ/θref)^n, i.e. k = k_ref·θref^(1-n); θref decorrelates k_ref and n, several starting exponents avoid local minima
    p_lin, _ = single_tau_fit(t, th, t_off_)
    ref, best = th.max() / 2, None
    for n0 in (0.8, 1.0, 1.3, 1.6, 2.0):
        res = least_squares(lambda p: simulate_loss(t, p[0], lambda x: p[1] * ref * (x / ref) ** p[2], t_off_) - th,
                            [p_lin[0], p_lin[1], n0], bounds=([1e-5, 1e-6, 0.3], [5, 1, 4]), diff_step=1e-3, x_scale="jac")
        if best is None or res.cost < best.cost:
            best = res
    return best.x, np.sqrt(np.mean(best.fun ** 2))

In [ ]:
class LossNet(nn.Module):
    # f(θ) = θ·softplus(h(θ)): zero at ambient temperature, positive above it
    def __init__(self, n=20):
        super().__init__()
        self.h = nn.Sequential(nn.Linear(1, n), nn.Tanh(), nn.Linear(n, n), nn.Tanh(), nn.Linear(n, 1))

    def forward(self, th):
        return th * nn.functional.softplus(self.h(th))

In [ ]:
class LossPINN(nn.Module):
    # temperature network with the switch input of Section 5, plus the learned loss law and the heating rate a
    def __init__(self, t_off_hat, n_units=100):
        super().__init__()
        self.t_off_hat = t_off_hat
        layers, d = [], 2
        for _ in range(4):
            layers += [nn.Linear(d, n_units), nn.Tanh()]
            d = n_units
        self.body = nn.Sequential(*layers, nn.Linear(n_units, 1))
        self.loss_law = LossNet()
        self.a = nn.Parameter(torch.tensor([1.0]))

    def forward(self, t):
        return self.body(torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1))

In [ ]:
def train_loss_pinn(t, th, t_off_, adam_epochs=8000, lbfgs_iters=1000, data_weight=1.0, physics_weight=1.0, seed=0, n_units=100, return_terms=False, log=None):
    torch.manual_seed(seed)
    t_scale, T_scale = float(t.max()), float(th.max())
    t_hat = np_to_th(t / t_scale); theta_hat = np_to_th(th / T_scale)
    model = LossPINN(t_off_ / t_scale, n_units=n_units).to(DEVICE)
    t_col = torch.linspace(0, 1, 800).view(-1, 1).to(DEVICE).requires_grad_(True)
    laser = (t_col < t_off_ / t_scale).float()

    def losses():                      # data loss and physics loss (ODE residual + initial condition)
        th_c = model(t_col)
        dth = grad(th_c, t_col)[0]
        res = dth - model.a * laser + model.loss_law(th_c)
        ic = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return ((model(t_hat) - theta_hat) ** 2).mean(), (res ** 2).mean() + ic

    def total_loss(phase):
        data_loss, physics_loss = losses()
        l = data_weight * data_loss + physics_weight * physics_loss
        l.backward()
        if log is not None:
            log.record(phase, data_loss, physics_loss)
        return l

    cache_path = cache_file("lossnet", t, th, t_off_, adam_epochs, lbfgs_iters, data_weight, physics_weight, seed, n_units)
    if USE_CACHE and log is None and os.path.exists(cache_path):
        model.load_state_dict(torch.load(cache_path, map_location=DEVICE))      # trained before: no training
    else:
        if log is not None:
            log.start(model, T_scale / t_scale)
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)
        for _ in range(adam_epochs):
            opt.zero_grad(); total_loss("Adam"); opt.step()
        if log is not None:
            log.end_adam()
        opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters, line_search_fn="strong_wolfe")
        def closure():
            opt2.zero_grad(); return total_loss("L-BFGS")
        opt2.step(closure)
        if log is not None:
            log.finish(opt2)
        if USE_CACHE and log is None:
            torch.save(model.state_dict(), cache_path)

    a_est = model.a.item() * T_scale / t_scale
    f_est = lambda x: model.loss_law(np_to_th(np.atleast_1d(x) / T_scale)).detach().cpu().numpy().ravel() * T_scale / t_scale
    fit = model(t_hat).detach().cpu().numpy().ravel() * T_scale
    if return_terms:   # the two losses at the end of training (before the weights)
        data_loss, physics_loss = losses()
        return a_est, f_est, fit, dict(data=data_loss.item(), physics=physics_loss.item())
    return a_est, f_est, fit

In [ ]:
# a_true (heating rate) is the global defined in Section 3
LOSS_LAWS = {
    "power n=1.25":     lambda x: (0.05 / 20 ** 1.25) * np.maximum(x, 0) ** 1.25,
    "linear+quadratic": lambda x: 1.25e-3 * np.maximum(x, 0) + 6.25e-5 * np.maximum(x, 0) ** 2,
}
TRUE_K = {"power n=1.25": {"k_power": 0.05 / 20 ** 1.25, "n": 1.25},
          "linear+quadratic": {"k1": 1.25e-3, "k2": 6.25e-5}}


def synthetic_case(law, t_off_, seed_):
    rng = np.random.default_rng(10 * seed_ + int(t_off_))
    t_ = np.arange(0, 2 * t_off_ + 1, 10.0)
    return t_, simulate_loss(t_, a_true, LOSS_LAWS[law], t_off_) + rng.normal(0, 0.1, t_.size)


def learned_f_rows(case, t_off_, seed_, th, a_n, f_n, p_s=(np.nan, np.nan), p_p=(np.nan, np.nan, np.nan)):
    theta = np.linspace(0.1, 1.0, 50) * th.max()
    return pd.DataFrame({"case": case, "t_off": t_off_, "seed": seed_, "theta": theta, "f": f_n(theta), "a_PINN": a_n,
                         "a_single": p_s[0], "k_single": p_s[1], "a_power": p_p[0],
                         "k_power": p_p[1] * (th.max() / 2) ** (1 - p_p[2]), "n_power": p_p[2]})


### 9.1 Synthetic check with a known loss law

In [ ]:
rows3, f_rows = [], []
for law, f_true in LOSS_LAWS.items():
    for t_off_ in (600.0, 1200.0):
        for seed_ in range(3):
            t_, th_ = synthetic_case(law, t_off_, seed_)
            a_r, _ = roper(t_, th_ + Tenv_true, t_off_)
            p_s, rms_s = single_tau_fit(t_, th_, t_off_)
            p_p, rms_p = power_law_fit(t_, th_, t_off_)
            a_n, f_n, fit_n = train_loss_pinn(t_, th_, t_off_, seed=seed_)
            f_rows.append(learned_f_rows(law, t_off_, seed_, th_, a_n, f_n, p_s, p_p))
            grid_ = np.linspace(1, th_.max(), 30)
            rows3.append(dict(law=law, t_off=t_off_, seed=seed_,
                              Roper=100 * (a_r - a_true) / a_true, single_tau=100 * (p_s[0] - a_true) / a_true,
                              power_law=100 * (p_p[0] - a_true) / a_true, loss_PINN=100 * (a_n - a_true) / a_true,
                              n_power=p_p[2], rms_single=rms_s, rms_power=rms_p,
                              rms_PINN=np.sqrt(np.mean((fit_n - th_) ** 2)),
                              f_err_pct=100 * np.max(np.abs(f_n(grid_) - f_true(grid_))) / f_true(th_.max())))

df3 = pd.DataFrame(rows3)
df3.to_csv("results/loss_law_synthetic.csv", index=False)
pd.concat(f_rows).to_csv("results/learned_f_synthetic.csv", index=False)
cols = ["Roper", "single_tau", "power_law", "loss_PINN", "n_power", "rms_single", "rms_power", "rms_PINN", "f_err_pct"]
print("=== Error in a [%], fitted exponent, RMS [°C], max error of the learned f [%] (mean over 3 seeds) ===")
print(df3.groupby(["law", "t_off"])[cols].mean().round(2).to_string())
print("\n=== Standard deviation over seeds ===")
print(df3.groupby(["law", "t_off"])[["Roper", "single_tau", "power_law", "loss_PINN"]].std().round(2).to_string())

### 9.2 Real curves

In [ ]:
DS_LABELS = ["DS1", "DS2a", "DS2b", "DS3", "DS4"]
rows4, loss_curves, f_rows = [], {}, []
for label, (ds_name, ds) in zip(DS_LABELS, REAL_DATASETS.items()):
    t_r, T_r = load_real(ds)
    th_r, t_off_r = T_r - ds["Tenv"], ds["t_off"]
    p_s, rms_s = single_tau_fit(t_r, th_r, t_off_r)
    p_p, rms_p = power_law_fit(t_r, th_r, t_off_r)
    runs = [train_loss_pinn(t_r, th_r, t_off_r, seed=s) for s in range(3)]
    f_rows += [learned_f_rows(label, t_off_r, s, th_r, r[0], r[1], p_s, p_p) for s, r in enumerate(runs)]
    a_runs = np.array([r[0] for r in runs])
    rms_runs = np.array([np.sqrt(np.mean((r[2] - th_r) ** 2)) for r in runs])
    grid_ = np.linspace(0.1, 1.0, 50) * th_r.max()
    loss_curves[ds_name] = (grid_ / th_r.max(), np.array([r[1](grid_) / grid_ for r in runs]))
    rows4.append(dict(dataset=ds_name.replace("\n", " "),
                      a_single=round(p_s[0], 5), a_power=round(p_p[0], 5), n_power=round(p_p[2], 2),
                      a_PINN=round(a_runs.mean(), 5), a_PINN_sd=round(a_runs.std(ddof=1), 5),
                      rms_single=round(rms_s, 2), rms_power=round(rms_p, 2), rms_PINN=round(rms_runs.mean(), 2),
                      r_ratio=round(np.mean([r[1](np.array([0.9 * th_r.max()]))[0] / r[1](np.array([0.2 * th_r.max()]))[0] * 0.2 / 0.9 for r in runs]), 2)))

df4 = pd.DataFrame(rows4)
df4.to_csv("results/loss_law_real.csv", index=False)
pd.concat(f_rows).to_csv("results/learned_f_real.csv", index=False)
print("a in °C/s; RMS in °C; r_ratio = r(0.9 θmax) / r(0.2 θmax) of the learned law (1 for a linear loss)")
print(df4.to_string(index=False))

# learned r(θ) normalised at 0.2 θmax, per curve (mean and range over the three seeds), saved for re-plotting
curve_rows = []
for label, (x, r_runs) in zip(DS_LABELS, loss_curves.values()):
    r_norm = r_runs / r_runs[:, [np.argmin(np.abs(x - 0.2))]]
    curve_rows.append(pd.DataFrame({"dataset": label, "theta_over_max": x, "r_mean": r_norm.mean(axis=0),
                                    "r_min": r_norm.min(axis=0), "r_max": r_norm.max(axis=0)}))
df_curves = pd.concat(curve_rows)
df_curves.to_csv("results/loss_law_curves.csv", index=False)

# fixed categorical order (validated reference palette) plus distinct line styles for black-and-white print
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
STYLES = ["-", "--", "-.", ":", (0, (5, 1, 1, 1, 1, 1))]
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(3.4, 2.3))
    for (label, d), color, ls in zip(df_curves.groupby("dataset", sort=False), COLORS, STYLES):
        ax.fill_between(d.theta_over_max, d.r_min, d.r_max, color=color, alpha=0.15, lw=0)
        ax.plot(d.theta_over_max, d.r_mean, color=color, ls=ls, lw=1.6, label=label)
    ax.axhline(1, color="0.45", lw=0.8)
    ax.set_xlabel(r"$\theta / \theta_{\max}$", fontsize=9)
    ax.set_ylabel(r"$r(\theta)\,/\,r(0.2\,\theta_{\max})$", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.3, lw=0.5)
    ax.legend(fontsize=7.5, frameon=False, ncol=2, loc="upper left")
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig5_learned_loss_law.jpg", dpi=200, bbox_inches="tight")
    fig.savefig("results/plots/fig5_learned_loss_law.pdf", bbox_inches="tight")
    plt.show()


### 9.3 Held-out extrapolation

In [ ]:
# Section 9.3: train on heating + first half of cooling, predict the second half of cooling
rows5 = []
for ds_name, ds in REAL_DATASETS.items():
    t_r, T_r = load_real(ds)
    th_r, t_off_r = T_r - ds["Tenv"], ds["t_off"]
    cut = t_off_r + 0.5 * (t_r.max() - t_off_r)
    fit, held = t_r <= cut, t_r > cut
    rms_held = lambda y: np.sqrt(np.mean((y[held] - th_r[held]) ** 2))

    p_s, _ = single_tau_fit(t_r[fit], th_r[fit], t_off_r)
    p_p, _ = power_law_fit(t_r[fit], th_r[fit], t_off_r)
    ref = th_r[fit].max() / 2                                   # theta_ref used inside power_law_fit
    y_s = simulate_loss(t_r, p_s[0], lambda x: p_s[1] * x, t_off_r)
    y_p = simulate_loss(t_r, p_p[0], lambda x: p_p[1] * ref * (x / ref) ** p_p[2], t_off_r)

    rms_pinn = []
    for seed_ in range(3):
        a_n, f_n, _ = train_loss_pinn(t_r[fit], th_r[fit], t_off_r, seed=seed_)
        y_n = simulate_loss(t_r, a_n, lambda x: float(f_n(np.array([x]))[0]), t_off_r)
        rms_pinn.append(rms_held(y_n))

    rows5.append(dict(dataset=ds_name.replace("\n", " "), n_fit=int(fit.sum()), n_held=int(held.sum()),
                      rms_single=rms_held(y_s), rms_power=rms_held(y_p),
                      rms_PINN=np.mean(rms_pinn), rms_PINN_sd=np.std(rms_pinn, ddof=1)))

df5 = pd.DataFrame(rows5)
df5.to_csv("results/loss_law_extrapolation.csv", index=False)
print("RMS error on the held-out cooling tail [°C]")
print(df5.round(2).to_string(index=False))

### 9.4 Fitted curves on the real data

In [ ]:
# Section 9.4: whole-curve fit, linear-loss PINN network output and learned-loss PINN network output on the real curves
from contextlib import redirect_stdout
import io

labels = ["DS1", "DS2a", "DS2b", "DS3", "DS4"]
fit_rows, summary = [], []
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.ravel()
axes[-1].set_visible(False)

for ax, label, (ds_name, ds) in zip(axes, labels, REAL_DATASETS.items()):
    t_r, T_r = load_real(ds)
    t_off_r, Tenv_r = ds["t_off"], ds["Tenv"]
    rms = lambda y: np.sqrt(np.mean((y - T_r) ** 2))

    a_c, tau_c = whole_curve_fit(t_r, T_r, t_off_r, Tenv_=Tenv_r)
    T_cf = heating_cooling(t_r, a_c, tau_c, t_off_r, Tenv_r)
    with redirect_stdout(io.StringIO()):                                # hide the per-epoch log
        a_l, tau_l, T_lin = train_photothermal(t_r, T_r, t_off_r, Tenv_=Tenv_r, return_fit=True)
    a_n, f_n, th_n = train_loss_pinn(t_r, T_r - Tenv_r, t_off_r, seed=0)
    T_learn = th_n + Tenv_r

    fit_rows.append(pd.DataFrame({"dataset": label, "time_s": t_r, "T_data": T_r, "T_whole_curve": T_cf,
                                  "T_linear_PINN_network": T_lin, "T_learned_PINN_network": T_learn}))
    summary.append(dict(dataset=label, a_whole_curve=round(a_c, 5), a_linear_PINN=round(a_l, 5), a_learned_PINN=round(a_n, 5),
                        rms_whole_curve=round(rms(T_cf), 3), rms_linear_PINN_network=round(rms(T_lin), 3),
                        rms_learned_PINN_network=round(rms(T_learn), 3)))

    ax.plot(t_r / 60, T_r, color='0.7', lw=3, label='data')
    ax.plot(t_r / 60, T_cf, color='tab:green', lw=1.6, label=f'whole-curve fit (RMS {rms(T_cf):.2f} °C)')
    ax.plot(t_r / 60, T_lin, color='tab:blue', ls=':', lw=2.2, label=f'linear-loss PINN, network (RMS {rms(T_lin):.2f} °C)')
    ax.plot(t_r / 60, T_learn, color='tab:red', ls='--', lw=1.6, label=f'learned-loss PINN, network (RMS {rms(T_learn):.2f} °C)')
    ax.axvline(t_off_r / 60, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_title(f"{label}: " + ds_name.replace("\n", " | "), fontsize=9)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('T (°C)')
    ax.legend(fontsize=7)

df_fits = pd.concat(fit_rows)
df_fits.to_csv("results/real_fits_curves.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))

plt.suptitle('Real data: fitted curves (PINN curves are the networks\' own outputs)', fontsize=11)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig6_real_fits.jpg', dpi=150, bbox_inches='tight')
plt.show()

### 9.5 Sensitivity to the weight of the physics loss

In [ ]:
# Section 9.5: sensitivity to the weight of the physics loss (physics_weight) (learned-loss PINN)
rows7 = []
for label, (ds_name, ds) in zip(["DS1", "DS2a", "DS2b", "DS3", "DS4"], REAL_DATASETS.items()):   # all five curves
    t_r, T_r = load_real(ds)
    Tenv_r, t_off_r = ds["Tenv"], ds["t_off"]
    th_r = T_r - Tenv_r
    p_s, _ = single_tau_fit(t_r, th_r, t_off_r)
    for w_ in (0.1, 1.0, 10.0):
        for seed_ in range(3):
            a_n, f_n, fit_n, tm = train_loss_pinn(t_r, th_r, t_off_r, data_weight=1.0, physics_weight=w_, seed=seed_, return_terms=True)
            y_n = simulate_loss(t_r, a_n, lambda x: float(f_n(np.array([x]))[0]), t_off_r)
            rows7.append(dict(dataset=label, model="learned-loss", physics_weight=w_, seed=seed_, a=a_n,
                              da_vs_single_pct=100 * (a_n - p_s[0]) / p_s[0],
                              rms_network=np.sqrt(np.mean((fit_n - th_r) ** 2)),
                              rms_ode=np.sqrt(np.mean((y_n - th_r) ** 2)),
                              data_loss=tm["data"], physics_loss=tm["physics"]))

df7 = pd.DataFrame(rows7)
df7.to_csv("results/loss_law_lambda_sensitivity_learned.csv", index=False)
print(df7.round(4).to_string(index=False))
print()
print(df7.groupby(["dataset", "model", "physics_weight"])[["da_vs_single_pct", "rms_network"]].agg(["min", "max"]).round(3).to_string())